# LUMIERE - exploratory data analysis

Longitudinal glioblastoma MRI with expert RANO evaluation
([Figshare collection 5904905](https://figshare.com/collections/The_LUMIERE_Dataset_Longitudinal_Glioblastoma_MRI_with_Expert_RANO_Evaluation/5904905)).

This notebook describes what the cohort actually contains, including what it
does **not** contain. Run `gbmplast acquire --dataset lumiere` first.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from gbmplast.adapters import get_adapter
from gbmplast.config import load_config
from gbmplast.report import dictionary as D
from gbmplast.report.theme import ROLE, apply_theme, series_color
from gbmplast.validate import validate_bundle

apply_theme()
config = load_config("lumiere")
adapter = get_adapter("lumiere", config)
print(adapter.describe().name)
print("raw dir:", adapter.raw_dir)

LUMIERE: Longitudinal Glioblastoma MRI with expert RANO evaluation
raw dir: /home/user/gbm_plasticity/data/raw/lumiere


## What the cohort releases, and what it does not

In [2]:
description = adapter.describe()
print("t0:", description.t0_definition)
print("\nKnown limitations:")
for item in description.known_limitations:
    print(" -", item)

t0: Week 0 = index surgery; timepoints are week-resolution.

Known limitations:
 - Overall survival is released in weeks with NO censoring indicator, so time-to-event modelling is not identifiable.
 - Timepoint resolution is weeks, not days.
 - No treatment records, no symptoms, no tissue specimens.
 - Segmentations are automated (DeepBraTumIA, HD-GLIO-AUTO), not manually curated.
 - 3 of 613 rated studies carry two conflicting expert ratings.
 - IDH is indeterminate ('sequencing required') for 10 patients.


In [3]:
adapter.availability()[["source_key", "relative_path", "required", "present",
                        "size_bytes"]]

,source_key,relative_path,required,present,size_bytes
0,demographics,LUMIERE-Demographics_Pathology.csv,True,True,5108
1,expert_rating,LUMIERE-ExpertRating-v202211.csv,True,True,31133
2,completeness,LUMIERE-datacompleteness.csv,True,True,23646
3,mrinfo,LUMIERE-MRinfo.csv,True,True,363872
4,radiomics_hdglio,LUMIERE-pyradiomics-hdglioauto-features.csv,True,True,13471846
5,radiomics_deepbratumia,LUMIERE-pyradiomics-deepbratumia-features.csv,True,True,21835161
6,readme,LUMIERE-readme.pdf,False,True,189228
7,imaging_zip,Imaging-v202211.zip,False,False,<NA>


## Canonical tables

In [4]:
bundle = adapter.build()
bundle.summary()

,table,n_rows,n_cols
0,patients,91,15
1,visits,655,22
2,imaging,3828,25
3,treatments,0,12
4,symptoms,0,15
5,molecular_samples,0,15
6,outcomes,702,14


In [5]:
for name, note in sorted(bundle.notes.items()):
    print(f"[{name}] {note}\n")

[molecular_samples] LUMIERE releases patient-level IDH/MGMT but no specimen-level records (no specimen ids, no collection times). Table empty by design; the markers live on 'patients'.

[survival] Overall survival has no censoring indicator; event_indicator is NA for all 91 patients.

[symptoms] LUMIERE releases no clinical symptoms. Table empty by design.

[treatments] LUMIERE releases no treatment records. The table is empty by design; treatment_phase on visits is inferred only from the Pre-Op/Post-Op expert ratings and is 'unknown' elsewhere.



Validation: structural checks on ids, ordering and survival identifiability.

In [6]:
report = validate_bundle(bundle)
report.to_frame()[["check", "table", "severity", "message", "n_affected"]]

,check,table,severity,message,n_affected
0,primary_key_unique,patients,info,primary key ['patient_id'] unique over 91 rows,0
1,primary_key_unique,visits,info,primary key ['visit_id'] unique over 655 rows,0
2,primary_key_unique,imaging,info,primary key ['imaging_id'] unique over 3828 rows,0
3,primary_key_unique,outcomes,info,primary key ['outcome_id'] unique over 702 rows,0
4,longitudinal_ordering,visits,info,visit_index consistent with time for all 91 pa...,0
5,survival_censoring,outcomes,warning,91 time-to-event rows have no censoring indica...,91


## Patients

In [7]:
patients = bundle.patients
print(f"{len(patients)} patients")
display_cols = ["sex", "idh_status", "mgmt_status"]
for col in display_cols:
    print(f"\n{col}:")
    print(patients[col].value_counts(dropna=False).to_string())

91 patients

sex:
sex
male      47
female    44

idh_status:
idh_status
wildtype         57
<NA>             23
indeterminate    10
mutant            1

mgmt_status:
mgmt_status
unmethylated    43
methylated      37
None            11


`IDH1 neg, Sequencing required` is an **inconclusive immunostain**, mapped to
`indeterminate` rather than to `wildtype`. Collapsing it into wild-type would
invent 10 definite calls.

In [8]:
patients.groupby(["idh_status_raw", "idh_status"], dropna=False).size()

idh_status_raw                 idh_status   
IDH1 neg, Sequencing required  indeterminate    10
R132H mut                      mutant            1
WT                             wildtype         56
wt                             wildtype          1
NaN                            NaN              23
dtype: int64

## Visits and the RANO course

In [9]:
visits = bundle.visits
print(f"{len(visits)} visits "
      f"({int(visits.has_imaging.sum())} imaged, "
      f"{int(visits.has_expert_rating.sum())} rated)")
print(f"rated but NOT imaged: "
      f"{int(((~visits.has_imaging) & visits.has_expert_rating).sum())}")
visits.rano_status.value_counts(dropna=False)

655 visits (638 imaged, 613 rated)
rated but NOT imaged: 17


rano_status
PD         254
Post-Op    122
SD          96
Pre-Op      92
<NA>        44
CR          27
PR          20
Name: count, dtype: int64

Three studies carry two expert ratings each. Two are genuine disagreements and
are flagged rather than silently resolved.

In [10]:
visits[visits.rano_conflict][["local_patient_id", "local_visit_id",
                              "rano_status", "rano_status_raw",
                              "rano_n_assessments"]]

,local_patient_id,local_visit_id,rano_status,rano_status_raw,rano_n_assessments
306,Patient-042,week-010,PD,PD | SD,2
327,Patient-043,week-106,PD,PD | Post-Op,2


In [11]:
per_patient = visits.groupby("patient_id").size()
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.hist(per_patient, bins=range(1, int(per_patient.max()) + 2),
        color=series_color(0), edgecolor="#fcfcfb")
ax.set_xlabel("visits per patient")
ax.set_ylabel("patients")
ax.set_title(f"Follow-up depth (median {per_patient.median():.0f}, "
             f"max {per_patient.max()})")
plt.show()

## Modality completeness

In [12]:
D.modality_completeness(bundle)

,cohort,modality,n_rows,n_available,n_unavailable,n_unknown,pct_available,n_visits_with_modality,n_patients_with_modality
0,lumiere,FLAIR,638,612,26,0,95.92,612,91
1,lumiere,SEG,1276,1198,78,0,93.89,599,91
2,lumiere,T1,638,617,21,0,96.71,617,91
3,lumiere,T1CE,638,632,6,0,99.06,632,91
4,lumiere,T2,638,626,12,0,98.12,626,91


## Radiomics: the absent-label question

A row exists for every (study, sequence, label), but when the segmentation
found no voxels of a label **every feature is NaN**. Those are volume zero,
not missing data - and they are concentrated in post-operative and
complete-response studies.

In [13]:
radiomics = adapter.load_radiomics("HD-GLIO-AUTO")
ct1 = radiomics[(radiomics.sequence == "CT1")
                & (radiomics.subregion == "enhancing")]
feature_cols = [c for c in ct1.columns if c.startswith("original_")]
absent = ct1[feature_cols].isna().all(axis=1)
print(f"{int(absent.sum())} of {len(ct1)} studies have NO enhancing label")

status = visits.set_index("visit_id").loc[
    ct1.loc[absent.to_numpy(), "visit_id"], "rano_status"]
status.value_counts(dropna=False)

175 of 599 studies have NO enhancing label


rano_status
Post-Op    48
SD         46
PD         38
CR         24
<NA>       13
PR          5
Pre-Op      1
Name: count, dtype: int64

## Longitudinal transitions

In [14]:
from gbmplast.longitudinal.build import build_longitudinal

result = build_longitudinal(adapter, config)
for line in result.provenance:
    print(" -", line)

 - segmentation: HD-GLIO-AUTO (labels verified from the released feature table)
 - primary sequence for intensity/texture: CT1
 - subregions: enhancing, non_enhancing
 - transitions: 477 over 81 patients
 - excluded: 87 index visits/patients (index visit has no imaging features x45; interval 0d below minimum 7d x39; interval 469d above maximum 365d x1)
 - predictors are computed from visits at or before the index visit; all next-visit quantities carry the 'next_' prefix and are outcomes
 - transitions whose next visit carries a RANO response assessment: 385 (247 PD)


In [15]:
table = result.table
table[["patient_id", "index_local_visit_id", "next_local_visit_id",
       "interscan_interval_days", "cur_enhancing__volume_mm3",
       "prev_enhancing__volume_mm3", "next_rano_status"]].head(10)

,patient_id,index_local_visit_id,next_local_visit_id,interscan_interval_days,cur_enhancing__volume_mm3,prev_enhancing__volume_mm3,next_rano_status
0,LUMIERE::Patient-001,week-000-2,week-044,308.0,0.0,13968.0,SD
1,LUMIERE::Patient-001,week-044,week-056,84.0,0.0,0.0,PD
2,LUMIERE::Patient-002,week-000,week-003,21.0,16826.0,NaN,Post-Op
3,LUMIERE::Patient-002,week-003,week-021,126.0,126.0,16826.0,SD
4,LUMIERE::Patient-002,week-021,week-037,112.0,0.0,126.0,PD
5,LUMIERE::Patient-002,week-037,week-040-1,21.0,4884.0,0.0,<NA>
6,LUMIERE::Patient-002,week-040-2,week-047,49.0,0.0,NaN,PD
7,LUMIERE::Patient-003,week-000-2,week-014,98.0,549.0,NaN,SD
8,LUMIERE::Patient-003,week-014,week-027,91.0,0.0,549.0,PD
9,LUMIERE::Patient-003,week-027,week-038,77.0,0.0,0.0,PD


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(table.interscan_interval_days, bins=30, color=series_color(0),
             edgecolor="#fcfcfb")
axes[0].set_xlabel("days between consecutive studies")
axes[0].set_ylabel("transitions")
axes[0].set_title("Inter-scan interval")

for i, subregion in enumerate(("enhancing", "non_enhancing")):
    values = pd.to_numeric(table[f"cur_{subregion}__volume_mm3"],
                           errors="coerce").dropna() / 1000.0
    axes[1].hist(values, bins=40, alpha=0.65,
                 color=ROLE.get(subregion, series_color(i)),
                 label=subregion.replace("_", "-"))
axes[1].set_xlabel("volume at the index visit (mL)")
axes[1].set_ylabel("transitions")
axes[1].set_title("Subregion volume")
axes[1].legend()
plt.tight_layout()
plt.show()

## The outcome

385 transitions have an expert RANO response assessment at the next visit.
The class balance (about 64% PD) reflects that RANO ratings were made on a
**selected** set of timepoints, not on every follow-up study - so this is not
the progression rate of routine surveillance imaging.

In [17]:
eligible = table[table.next_rano_is_response_assessment == True]  # noqa: E712
print(f"{len(eligible)} eligible transitions, "
      f"{eligible.patient_id.nunique()} patients")
eligible.next_rano_status.value_counts()

385 eligible transitions, 78 patients


next_rano_status
PD    247
SD     92
CR     26
PR     20
Name: count, dtype: int64

## Survival is not modellable here

The cohort releases a survival time in weeks with **no vital-status column**,
so every patient would be treated as having died. The pipeline refuses to fit
a Cox model on this cohort.

In [18]:
from gbmplast.modeling.survival import can_fit_survival

print(can_fit_survival(bundle, "overall_survival"))

(False, 'overall_survival is released without a censoring indicator, so a Cox model is not identifiable: every patient would be treated as having had the event. Refusing to fit.')
